# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL: [https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # Do not subscript or iterate over this object.

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and entity `@id`s.

Here, we inspect the record sets, then for each, print associated fields and columns with their `@id`s.

In [ ]:
from pprint import pprint

# Record set IDs from the Croissant schema.
record_sets = [r['@id'] for r in dataset.metadata.to_json().get('recordSet', [])]

print(f"Found {len(record_sets)} record set(s).\n")
for record_set_id in record_sets:
    # Get the record set by ID
    rs = None
    for r in dataset.metadata.to_json()['recordSet']:
        if r['@id'] == record_set_id:
            rs = r
            break
    print(f"Record set @id: {record_set_id}")
    print(f"  name: {rs.get('name','')} (")
    # List fields
    if 'field' in rs:
        print("  Fields:")
        for f in rs['field']:
            if isinstance(f, dict):
                # Inline field
                print(f"    - @id: {f.get('@id','')} name: {f.get('name','')}")
            else:
                print(f"    - {f}")
    # List columns
    if 'column' in rs:
        print("  Columns:")
        for c in rs['column']:
            if isinstance(c, dict):
                print(f"    - @id: {c.get('@id','')} name: {c.get('name','')}")
            else:
                print(f"    - {c}")
    print()

# If there are no record sets, display a helpful tip for the user.
if len(record_sets) == 0:
    print("No record sets found in the provided dataset schema. Please review the Croissant metadata or contact the dataset curator.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. 
Use the record set, field, and column `@id`s from the overview above.

> **Note:** If you identified record set(s) in the previous cell, substitute the record set `@id` below. If not, this section will not run and you may need to manually discover or guess from metadata.

In [ ]:
# Example pattern for record set extraction
dataframes = {}

# If the dataset has record sets, extract them by @id
if len(record_sets) > 0:
    for record_set_id in record_sets:
        try:
            print(f"Extracting records for {record_set_id}")
            records = list(dataset.records(record_set=record_set_id))
            dataframes[record_set_id] = pd.DataFrame(records)
            print(f"Shape: {dataframes[record_set_id].shape}")
        except Exception as e:
            print(f"Could not extract {record_set_id}: {e}")
    # Display columns from the first extracted dataframe
    if dataframes:
        first_rs = list(dataframes.keys())[0]
        print(f"Columns for record set {first_rs}: {dataframes[first_rs].columns.tolist()}")
        display(dataframes[first_rs].head())
else:
    print("No record sets to extract records from.")

## 4. Exploratory Data Analysis (EDA)
Apply standard EDA steps, such as filtering records, normalizing data, and grouping. 
Reference columns and fields **always by their `@id`**. 

Below we select a numeric field (by `@id`) for demonstration.

In [ ]:
# --- Replace these @id values with those printed in Overview above ---
# Example:
# record_set_id = 'cr:RecordSet/YourRecordSetId'
# numeric_field_id = 'cr:Field/YourNumericFieldId'
# group_field_id = 'cr:Field/YourGroupFieldId'

# Attempt best-guess on the most likely IDs given schema subject (modify as needed after inspection):
if dataframes:
    record_set_id = list(dataframes.keys())[0]
    df = dataframes[record_set_id]
    print(f"Available columns in {record_set_id}:")
    print(df.columns.tolist())
    # Best-guess the first numeric field (usually abundance or coverage, e.g. 'abundance', 'coverage')
    possible_numeric_fields = [col for col in df.columns if df[col].dtype in [int, float]]
    if not possible_numeric_fields:
        # Try to infer by name
        numeric_field_id = None
        for col in df.columns:
            if 'abundance' in col.lower() or 'coverage' in col.lower() or 'count' in col.lower() or 'MW' in col:
                numeric_field_id = col
                break
    else:
        numeric_field_id = possible_numeric_fields[0]

    if numeric_field_id is not None:
        print(f"Using numeric field for demo: {numeric_field_id}")
        # Filter records for demonstration
        threshold = df[numeric_field_id].mean()  # Use mean as example threshold
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
        print(filtered_df.head())

        # Normalize selected field
        filtered_df[f"{numeric_field_id}_normalized"] = (
            (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
            filtered_df[numeric_field_id].std()
        )
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Try to group by a likely group field (e.g. experimental condition or 'description')
        group_field_id = None
        for col in df.columns:
            if col.lower() in ['sample', 'group', 'condition', 'description', 'modification']:
                group_field_id = col
                break
        if group_field_id:
            grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
            print(f"Grouped data by {group_field_id}:")
            print(grouped_df.head())
        else:
            print("No suitable group field found for grouping.")
    else:
        print("No suitable numeric field detected for EDA.")
else:
    print("No dataframes available for EDA. Check previous extraction step.")

## 5. Visualization
Visualize data distributions or relationships between fields.

Below, we use matplotlib to plot the normalized numeric field distribution, if possible.

In [ ]:
import matplotlib.pyplot as plt

if 'filtered_df' in locals() and numeric_field_id is not None:
    filtered_df[f"{numeric_field_id}_normalized"].hist(bins=25)
    plt.title(f"Distribution of normalized {numeric_field_id}")
    plt.xlabel(f"{numeric_field_id}_normalized")
    plt.ylabel("Count")
    plt.show()
else:
    print("Nothing to visualize: ensure EDA step ran successfully.")

## 6. Conclusion
We have loaded and explored the FAIR\$^2$ dataset package using `mlcroissant`. The notebook guided you through examining record set structure, extracting tabular data, filtering and normalizing numeric results, and visualizing outcomes. 

You can use this structure to further analyze or model the protein abundance and modification data from extracellular vesicle experiments.